<a href="https://colab.research.google.com/github/GiuseppeMariani1/KingsInvestmentFund/blob/Launchers/KIF_Streamlit_cloudfare_launcher.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Clone + install
!rm -rf KingsInvestmentFund
!git clone https://github.com/mattyyychan/KingsInvestmentFund.git
!pip -q install -r /content/KingsInvestmentFund/requirements.txt
!pip -q install cloudflared


Cloning into 'KingsInvestmentFund'...
remote: Enumerating objects: 62, done.
remote: Counting objects: 100% (62/62), done.
remote: Compressing objects: 100% (59/59), done.
remote: Total 62 (delta 6), reused 55 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (62/62), 123.34 KiB | 1.10 MiB/s, done.
Resolving deltas: 100% (6/6), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 66.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 134.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 109.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 kB 4.3 MB/s eta 0:00:00


In [ ]:
# Enter  WRDS creds (session-only)
import os, getpass
os.environ["WRDS_USERNAME"] = input("WRDS username: ").strip()
os.environ["WRDS_PASSWORD"] = getpass.getpass("WRDS password: ")


In [3]:
from pathlib import Path

path = Path("/content/KingsInvestmentFund/investment_dashboard/dashboard")
path.mkdir(parents=True, exist_ok=True)

(path / "wrds_auth.py").write_text("""
from pathlib import Path
import os

WRDS_HOST = "wrds-pgdata.wharton.upenn.edu"
WRDS_PORT = "9737"
WRDS_DB   = "wrds"

def ensure_pgpass(username: str, password: str) -> None:
    pgpass = Path.home() / ".pgpass"
    line = f"{WRDS_HOST}:{WRDS_PORT}:{WRDS_DB}:{username}:{password}\\n"
    if pgpass.exists():
        existing = pgpass.read_text(errors="ignore")
        if line in existing:
            return
        pgpass.write_text(existing + ("" if existing.endswith("\\n") else "\\n") + line)
    else:
        pgpass.write_text(line)
    os.chmod(pgpass, 0o600)
""")

print("wrds_auth.py written")


wrds_auth.py written


In [4]:
from pathlib import Path
import sys

# Fix: add dashboard folder to sys.path
sys.path.append("/content/KingsInvestmentFund/investment_dashboard/dashboard")

app_path = Path("/content/KingsInvestmentFund/investment_dashboard/dashboard/app.py")
txt = app_path.read_text()

marker = "# --- WRDS auth (Colab/Deploy friendly) ---"
if marker not in txt:
    needle = "import streamlit as st"
    insert = (
        "import os\n"
        "from wrds_auth import ensure_pgpass\n\n"
        "# --- WRDS auth (Colab/Deploy friendly) ---\n"
        "WRDS_USER = None\n"
        "WRDS_PASS = None\n"
        "try:\n"
        "    WRDS_USER = st.secrets.get('wrds', {}).get('username')\n"
        "    WRDS_PASS = st.secrets.get('wrds', {}).get('password')\n"
        "except Exception:\n"
        "    pass\n"
        "WRDS_USER = WRDS_USER or os.getenv('WRDS_USERNAME')\n"
        "WRDS_PASS = WRDS_PASS or os.getenv('WRDS_PASSWORD')\n"
        "if WRDS_USER and WRDS_PASS:\n"
        "    ensure_pgpass(WRDS_USER, WRDS_PASS)\n"
        "# ----------------------------------------\n"
    )
    app_path.write_text(txt.replace(needle, needle + "\n" + insert))

print("patched:", marker in app_path.read_text())


patched: True


In [6]:
# Go to dashboard folder
%cd /content/KingsInvestmentFund/investment_dashboard/dashboard

# Run Streamlit in background
!streamlit run app.py --server.port 8501 --server.address 0.0.0.0 --server.headless true &>/content/streamlit.log &

# Download official Cloudflare binary (works in Colab)
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64

# Run tunnel
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501 --no-autoupdate


/content/KingsInvestmentFund/investment_dashboard/dashboard
2026-02-05T14:04:45Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-02-05T14:04:45Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-02-05T14:04:48Z INF +--------------------------------------------------------------------------------------------+
2026-02-05T14:04:48Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-02-05T14:04:48Z 